# End-to-End ML Demo: Trade Anomaly Detection

This notebook demonstrates Snowflake's complete ML lifecycle:
- **Feature Store**: Entities and Feature Views
- **Dataset**: Versioned training data snapshots
- **Experiments**: Track and compare model training runs
- **Model Registry**: Log and deploy models
- **ML Observability**: Monitor model performance
- **ML Lineage**: Trace data flow

## 1. Setup & Configuration

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np

session = get_active_session()

DATABASE = "FSI_DEMO_DB"
SCHEMA = "CAPITAL_MARKETS"
WAREHOUSE = "FSI_DEMO_WH"

session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()
session.sql(f"USE WAREHOUSE {WAREHOUSE}").collect()

print(f"Connected: {DATABASE}.{SCHEMA}")
print(f"Warehouse: {WAREHOUSE}")

## 2. Feature Store

Create an Entity (the key) and Feature View (transformations).

In [ ]:
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode

fs = FeatureStore(
    session=session,
    database=DATABASE,
    name=SCHEMA,
    default_warehouse=WAREHOUSE,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

trade_entity = Entity(name="TRADE_ENTITY", join_keys=["TRADE_ID"])
fs.register_entity(trade_entity)

print("Feature Store initialized")
print(f"Entity registered: TRADE_ENTITY")

In [ ]:
trade_features_df = session.sql("""
    SELECT 
        TRADE_ID,
        QUANTITY,
        PRICE,
        GROSS_AMOUNT,
        COMMISSION,
        CASE WHEN TRADE_TYPE = 'BUY' THEN 1 ELSE 0 END AS IS_BUY,
        CASE WHEN IS_ALGORITHM_TRADE THEN 1 ELSE 0 END AS IS_ALGO,
        EXTRACT(HOUR FROM TRADE_DATE) AS TRADE_HOUR,
        COMMISSION / NULLIF(GROSS_AMOUNT, 0) AS COMMISSION_RATE,
        CASE WHEN ANOMALY_FLAG THEN 1 ELSE 0 END AS LABEL
    FROM TRADES
""")

trade_fv = FeatureView(
    name="TRADE_ANOMALY_FEATURES",
    entities=[trade_entity],
    feature_df=trade_features_df,
    refresh_freq="1 hour",
    desc="Trade features for anomaly detection"
)

trade_fv = fs.register_feature_view(
    feature_view=trade_fv,
    version="v1",
    block=True,
    overwrite=True
)

print(f"Feature View registered: {trade_fv.name} v{trade_fv.version}")

## 3. Create Dataset

Create an immutable, versioned snapshot for reproducible training.

In [ ]:
spine_df = session.table("TRADES").select("TRADE_ID")

ds = fs.generate_dataset(
    name="TRADE_ANOMALY_DATASET",
    spine_df=spine_df,
    features=[trade_fv],
    version="v1",
    spine_label_cols=["LABEL"],
    desc="Trade anomaly detection training dataset"
)

training_df = ds.read.to_snowpark_dataframe()
print(f"Dataset created: {ds.fully_qualified_name}")
print(f"Version: {ds.selected_version.name}")
print(f"Training data: {training_df.count()} rows")

## 4. Prepare Training Data

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pdf = training_df.to_pandas()

feature_cols = ['QUANTITY', 'PRICE', 'GROSS_AMOUNT', 'COMMISSION', 
                'IS_BUY', 'IS_ALGO', 'TRADE_HOUR', 'COMMISSION_RATE']

X = pdf[feature_cols].fillna(0)
y = pdf['LABEL']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training: {X_train.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")
print(f"Anomaly rate: {y.mean():.2%}")

## 5. Experiment Tracking

Track hyperparameters and metrics across training runs.

In [ ]:
from snowflake.ml.experiment import ExperimentTracking

exp = ExperimentTracking(session=session)
exp.set_experiment("TRADE_ANOMALY_EXPERIMENT")

print("Experiment ready: TRADE_ANOMALY_EXPERIMENT")

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import datetime

params_v1 = {
    "n_estimators": 100,
    "max_depth": 5,
    "learning_rate": 0.1,
    "random_state": 42
}

run_suffix = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
with exp.start_run(f"GBM_RUN_1_{run_suffix}"):
    exp.log_params(params_v1)
    
    model_v1 = GradientBoostingClassifier(**params_v1)
    model_v1.fit(X_train_scaled, y_train)
    
    y_pred = model_v1.predict(X_test_scaled)
    y_proba = model_v1.predict_proba(X_test_scaled)[:, 1]
    
    metrics_v1 = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba)
    }
    exp.log_metrics(metrics_v1)
    
    print("Run 1 Complete:")
    for k, v in metrics_v1.items():
        print(f"  {k}: {v:.4f}")

In [ ]:
params_v2 = {
    "n_estimators": 200,
    "max_depth": 7,
    "learning_rate": 0.05,
    "random_state": 42
}

run_suffix = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
with exp.start_run(f"GBM_RUN_2_{run_suffix}"):
    exp.log_params(params_v2)
    
    model_v2 = GradientBoostingClassifier(**params_v2)
    model_v2.fit(X_train_scaled, y_train)
    
    y_pred_v2 = model_v2.predict(X_test_scaled)
    y_proba_v2 = model_v2.predict_proba(X_test_scaled)[:, 1]
    
    metrics_v2 = {
        "accuracy": accuracy_score(y_test, y_pred_v2),
        "precision": precision_score(y_test, y_pred_v2, zero_division=0),
        "recall": recall_score(y_test, y_pred_v2, zero_division=0),
        "f1_score": f1_score(y_test, y_pred_v2, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba_v2)
    }
    exp.log_metrics(metrics_v2)
    
    print("Run 2 Complete:")
    for k, v in metrics_v2.items():
        print(f"  {k}: {v:.4f}")

## 6. Model Registry

Log the best model for deployment and SQL inference.

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)

best_model = model_v2 if metrics_v2["f1_score"] >= metrics_v1["f1_score"] else model_v1
best_metrics = metrics_v2 if metrics_v2["f1_score"] >= metrics_v1["f1_score"] else metrics_v1

sample_input = training_df.select(feature_cols).limit(10)

mv = registry.log_model(
    model=best_model,
    model_name="TRADE_ANOMALY_DETECTOR",
    version_name="V1",
    sample_input_data=sample_input,
    metrics=best_metrics,
    comment="GradientBoosting classifier for trade anomaly detection"
)

print(f"Model registered: {mv.model_name} v{mv.version_name}")
print(f"Best F1 Score: {best_metrics['f1_score']:.4f}")

In [ ]:
test_df = pd.DataFrame(X_test_scaled[:5], columns=feature_cols)
predictions = best_model.predict(test_df)

print("Sample Predictions:")
result_df = test_df.copy()
result_df["PREDICTION"] = predictions
result_df

## 7. ML Observability

Create inference log table and model monitor for drift detection.

In [ ]:
inference_df = pd.DataFrame({
    'ID': range(len(X_test)),
    'TIMESTAMP': pd.Timestamp.now(),
    'PREDICTION': y_pred_v2,
    'ACTUAL': y_test.values
})

for i, col in enumerate(feature_cols):
    inference_df[col] = X_test_scaled[:, i]

session.create_dataframe(inference_df).write.mode("overwrite").save_as_table(
    "TRADE_ANOMALY_INFERENCE_LOG"
)

print(f"Inference log created: {len(inference_df)} rows")

In [ ]:
%%sql -r dataframe_1
CREATE OR REPLACE MODEL MONITOR TRADE_ANOMALY_MONITOR
WITH 
    MODEL = TRADE_ANOMALY_DETECTOR
    VERSION = 'V1'
    FUNCTION = 'PREDICT'
    SOURCE = TRADE_ANOMALY_INFERENCE_LOG
    TIMESTAMP_COLUMN = TIMESTAMP
    WAREHOUSE = FSI_DEMO_WH
    REFRESH_INTERVAL = '1 hour'
    AGGREGATION_WINDOW = '1 day'
    ID_COLUMNS = ('ID')
    PREDICTION_CLASS_COLUMNS = ('PREDICTION')
    ACTUAL_CLASS_COLUMNS = ('ACTUAL')

In [ ]:
%%sql -r dataframe_2
SHOW MODEL MONITORS

## 8. ML Lineage

View in Snowsight: **AI & ML → Models → TRADE_ANOMALY_DETECTOR → Lineage**

In [ ]:
%%sql -r dataframe_3
SELECT * FROM TABLE(SNOWFLAKE.CORE.GET_LINEAGE(
    'FSI_DEMO_DB.CAPITAL_MARKETS.TRADE_ANOMALY_DETECTOR',
    'MODULE',
    'UPSTREAM',
    3,
    'V1'
))

In [ ]:
print("="*50)
print("ML DEMO COMPLETE")
print("="*50)
print(f"""
Feature Store:
  - Entity: TRADE_ENTITY
  - Feature View: TRADE_ANOMALY_FEATURES (v1)

Dataset: TRADE_ANOMALY_DATASET (v1)

Experiment: TRADE_ANOMALY_EXPERIMENT
  - GBM_RUN_1: F1={metrics_v1['f1_score']:.4f}
  - GBM_RUN_2: F1={metrics_v2['f1_score']:.4f}

Model: TRADE_ANOMALY_DETECTOR (V1)

Monitor: TRADE_ANOMALY_MONITOR
""")
print("="*50)

## 9. Cleanup (Optional)

Run to delete all resources and re-run notebook fresh.

In [ ]:
print("Cleaning up...")

cleanup_sql = [
    "DROP MODEL MONITOR IF EXISTS TRADE_ANOMALY_MONITOR",
    "DROP TABLE IF EXISTS TRADE_ANOMALY_INFERENCE_LOG",
    "DROP MODEL IF EXISTS TRADE_ANOMALY_DETECTOR",
    "DROP DATASET IF EXISTS TRADE_ANOMALY_DATASET",
    'DROP VIEW IF EXISTS "TRADE_ANOMALY_FEATURES$v1"',
    "DROP TAG IF EXISTS SNOWML_FEATURE_STORE_ENTITY_TRADE_ENTITY",
    "DROP TAG IF EXISTS TRADE_ENTITY",
    "DROP TAG IF EXISTS SNOWML_FEATURE_STORE_OBJECT",
    "DROP TAG IF EXISTS SNOWML_FEATURE_VIEW_METADATA"
]

for sql in cleanup_sql:
    try:
        session.sql(sql).collect()
        obj = sql.split()[-1]
        print(f"  Dropped: {obj}")
    except Exception as e:
        pass

try:
    exp.delete_experiment("TRADE_ANOMALY_EXPERIMENT")
    print("Dropped: TRADE_ANOMALY_EXPERIMENT")
except:
    pass

print("Cleanup complete!")